In [1]:
# standard library
import sys
from pathlib import Path

# third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
    HistGradientBoostingClassifier
)

from src.evaluation import evaluate_classification_model, check_overfitting
from src.data_prep import (
    to_snake_case,
    unzip_csv,
    find_duplicates,
    remove_duplicates,
    compare_groups_mannwhitney,
    compare_group_centroids
)
from src.visualization import (
    plot_categorical_distribution,
    plot_numeric_distribution,
    plot_correlation_heatmap,
    plot_group_means_heatmap,
    plot_centroid_distances,
    plot_effect_size_comparison,
    check_float_binary_columns
)

# Diabetes prediction

## Introduction

Diabetes is one of the most common chronic diseases worldwide, and a large share of cases remain undiagnosed until complications have already developed. Identifying at-risk individuals earlier - from routinely available lifestyle, demographic, and health indicators - could support more timely screening and intervention, potentially at lower cost than relying solely on clinical lab testing.

This project builds and compares diabetes risk prediction models on two independent datasets: a large-scale U.S. behavioral health survey (BRFSS 2015) containing lifestyle and demographic indicators, and a smaller clinical-style dataset that additionally includes direct biomarker measurements. Working with both allows a direct comparison between what is achievable from lifestyle/demographic data alone versus data that includes clinical diagnostic indicators, and highlights the practical trade-offs between the two approaches.

The analysis follows a full pipeline - data cleaning, exploratory data analysis, preprocessing, and model training and evaluation - with particular attention to identifying and correcting limitations in the underlying data along the way.

## Diabetes Prediction - Project Overview

This project compares two diabetes datasets that differ substantially in what kind of signal they offer a model.

The Diabetes prediction dataset (~100k rows) is a clinically-oriented dataset: alongside demographic and lifestyle fields, it includes lab-derived measurements - `HbA1c_level` and `blood_glucose_level` - that are themselves part of the official diagnostic criteria for diabetes.

The Diabetes Health Indicators Dataset, derived from the CDC's BRFSS 2015 survey (~250k+ rows), contains no lab measurements at all - only self-reported lifestyle, demographic, and comorbidity indicators (blood pressure, cholesterol, general health, physical activity, income, education, etc.), making it a harder and arguably more realistic task: predicting diabetes risk purely from indirect, survey-based population health data rather than direct clinical readings.

Training both datasets side by side makes this contrast explicit: the first shows what's achievable when direct clinical markers are available, while the second shows what's achievable when they aren't - which is the more relevant scenario for large-scale, low-cost screening.

## Dataset Overview - Diabetes prediction dataset (Clinical)

Some attributes like age and gender are self-explanatory. However, attributes like HbA1c_level, blood_glucose_level, and even bmi require domain knowledge, which is why we go through some explanations below.

**BMI (Body Mass Index)**

bmi: Body Mass Index. A generally useful metric, calculated as weight (kg) / height (m)$^2$. However, it does not account for age, sex, muscle mass, or body fat percentage.

Standard World Health Organization classification:
- Underweight: < 18.5
- Normal weight: 18.5 – 24.9
- Overweight: 25.0 – 29.9
- Obese: 30.0+

**HbA1c_level (Glycated Hemoglobin)**

Reflects the average blood sugar level over the past 2–3 months, measured as a percentage. Unlike a single glucose reading, it is not affected by short-term fluctuations (e.g. what the person ate that morning), which makes it a more stable diagnostic indicator.

Clinical thresholds:
- Normal: < 5.7%
- Prediabetes: 5.7% – 6.4%
- Diabetes: $\geq$ 6.5%

**blood_glucose_level**

A single snapshot measurement of sugar concentration in the blood at the time of testing, expressed in mg/dL (milligrams of glucose per deciliter of blood) - unlike HbA1c which reflects a longer-term average. Depending on whether the reading was fasting or random, the diagnostic thresholds differ:

Fasting glucose:
- Normal: < 100 mg/dL
- Prediabetes: 100 – 125 mg/dL
- Diabetes: $\geq$ 126 mg/dL

Random (non-fasting) glucose:
- Diabetes: $ \geq $ 200 mg/dL

> Note: this dataset does not specify whether readings are fasting or random, which is a limitation worth mentioning in the analysis.

**smoking_history**

Categorical variable with 6 possible values: `never`, `former`, `current`, 
`not current`, `ever`, and `No Info`.

The `No Info` category is not a smoking status - it indicates missing data (the respondent's smoking history was not recorded). Treating it as its own category, rather than dropping it, avoids losing rows, but it should not be interpreted as "never smoked."

The overlap between `former`, `not current`, and `ever` is somewhat ambiguous in the source data and worth noting as a limitation - these may need to be consolidated during preprocessing.

**hypertension** and **heart_disease**

Both are binary indicators (0 = No, 1 = Yes) representing whether the respondent has been previously diagnosed with high blood pressure or heart disease, respectively. These are included as established comorbidity risk factors for type 2 diabetes - hypertension and cardiovascular disease frequently co-occur with diabetes due to shared risk factors such as obesity and metabolic syndrome.

**diabetes** (target variable)

Binary label (0 = No diabetes, 1 = Diabetes). Note the overlap with the clinical thresholds discussed above - since HbA1c $ \geq $ 6.5% and fasting glucose $ \geq $ 126 mg/dL are themselves diagnostic criteria for diabetes, these two features are likely to be extremely strong (possibly near-deterministic) predictors of the target. This should be flagged explicitly, as models may achieve very high accuracy simply by learning the clinical cutoff rather than genuine risk patterns from lifestyle/demographic features.

## Dataset Overview - Diabetes Health Indicators Dataset (BRFSS Survey)

This dataset is derived from the CDC's 2015 Behavioral Risk Factor Surveillance System (BRFSS) survey. Unlike the smaller diabetes prediction dataset, it contains no lab measurements - every feature is either self-reported or derived from a survey question, so domain knowledge is needed to interpret several of them correctly.

**general_health**
Self-reported overall health on a 5-point ordinal scale (1 = Excellent to 5 = Poor). It is a subjective measure, but consistently one of the strongest predictors of diabetes status across BRFSS-based studies, likely because it implicitly captures a wide range of unmeasured health conditions.

**high_blood_pressure** and **high_cholesterol**
Binary indicators (0 = No, 1 = Yes) for a prior diagnosis of hypertension or high cholesterol, respectively. Both are established comorbidities of type 2 diabetes, frequently co-occurring due to shared risk factors such as obesity and metabolic syndrome.

**chol_checked_recently**
Binary indicator of whether the respondent had their cholesterol checked within the last 5 years. This is not a risk factor in the clinical sense - it is a healthcare-engagement proxy. People who are never screened are also less likely to be screened and diagnosed for diabetes, so this feature can pick up a detection-bias signal rather than a genuine physiological one. This should be flagged explicitly, since a model may lean on it for reasons unrelated to real diabetes risk.

**bmi**
Body Mass Index, calculated as weight (kg) / height (m)$^2$. Same World Health Organization classification as in the other dataset:
- Underweight: < 18.5
- Normal weight: 18.5 – 24.9
- Overweight: 25.0 – 29.9
- Obese: 30.0+

**physical_health_bad_days** and **mental_health_bad_days**
Self-reported number of days (0–30) in the past month during which the respondent's physical or mental health was "not good." These are continuous proxies for general health burden rather than diabetes-specific indicators.

**difficulty_walking**
Binary indicator (0 = No, 1 = Yes) for serious difficulty walking or climbing stairs - a proxy for mobility limitations, which can be both a cause (reduced physical activity) and a consequence (neuropathy, obesity) of diabetes.

**had_stroke** and **heart_disease_or_attack**
Binary indicators for a prior diagnosis of stroke or coronary heart disease/myocardial infarction. Like hypertension and high cholesterol, these are established diabetes comorbidities rather than direct predictors.

**physically_active**, **consumes_fruit_daily**, **consumes_veggies_daily**, **heavy_alcohol_consumption**, **smoked_at_least_100_cigarettes**
Binary lifestyle indicators self-reported by the respondent (e.g. "ever smoked at least 100 cigarettes" is the standard BRFSS proxy for "ever a smoker"). These carry real signal but are individually weak, indirect predictors compared to the comorbidity and general-health features above.

**skipped_doctor_due_to_cost**
Binary indicator of whether cost prevented the respondent from seeing a doctor when needed in the past year - a healthcare-access proxy that, similarly to `chol_checked_recently`, can reflect socioeconomic and detection-bias effects rather than physiological risk.

**income_level** and **education_level**
Ordinal categorical variables (grouped income brackets and education levels). Both are socioeconomic proxies, correlated with diet, healthcare access, and diagnosis rates rather than direct physiological risk factors.

**age** and **sex**
`age` is reported in BRFSS as a grouped ordinal category (13 age brackets) rather than an exact value, which should be kept in mind when interpreting model coefficients or feature importances. `sex` is binary (0/1).

**diabetes** (target variable)
Binary label (0 = No diabetes, 1 = Prediabetes, 2 = Diabetes), with a substantial class imbalance (~8.8% positive). Because none of the features here are direct lab measurements, no single feature is expected to be near-deterministic the way `HbA1c_level` or `blood_glucose_level` are in the other dataset - predictive performance instead comes from combining many weak, indirect signals, which is the main reason overall metrics (ROC-AUC, PR-AUC) are noticeably lower here than on the lab-based dataset.

## Conclusion

### Methodology Deviations from the Baseline Pipeline

For the `BRFSS` dataset, the original public methodology (Teboul et al.) was reproduced with two deliberate, validated deviations: `_BMI5` was kept at its original decimal precision rather than rounded to the nearest integer, and `_AGE80` (single-year age) was used instead of `_AGEG5YR` (5-year age bins), with rows lacking a reported age (`_AGEG5YR == 14`) still excluded to avoid using CDC-imputed rather than reported values. These two changes were empirically validated: the rate of full duplicate rows dropped from 9.42% to 0.47%, and conflicting duplicates (identical features, different diabetes label) dropped from 10.16% to 0.49% - a roughly 20 times reduction confirming that coarse feature resolution in the original methodology was a primary driver of label ambiguity.

The target variable itself was also reconstructed differently from the official methodology. Rather than merging prediabetes with the non-diabetic class (as the public `Diabetes_binary` dataset does), prediabetes was merged with the diabetic class, based on three independent lines of evidence: a feature-mean comparison (15 of 18 features placed prediabetes closer to diabetes than to no-diabetes), Mann-Whitney effect sizes (consistently 2-4x larger against no-diabetes than against diabetes), and centroid distance in standardized feature space (0.916 vs. 0.407). This is a case where the data contradicted the established public methodology, and the deviation is explicitly justified rather than assumed.

Near-constant features identified during EDA (`has_healthcare_coverage`, `chol_checked_recently`, `had_stroke`) were not removed automatically based on prevalence alone. A crosstab against the target showed `had_stroke` and `chol_checked_recently` carry meaningful predictive signal despite low or near-universal prevalence, and only `has_healthcare_coverage` was removed. `chol_checked_recently` was flagged as likely reflecting a detection/healthcare-engagement effect rather than a genuine risk factor, and is not interpreted causally despite its predictive value.

### Data Quality Findings

Both datasets showed evidence of data quality limitations that shaped the analysis. In the `iammustafatz` dataset, `bmi` showed an artificial concentration around a single value (27.32, ~22.5% of all rows) alongside extreme outliers (max = 97.65), and `HbA1c_level`/`blood_glucose_level` were found to take only 18 discrete values each, indicating quantized or synthetically generated data rather than raw clinical measurements.

Most significantly, `HbA1c_level` and `blood_glucose_level` were identified as the clinical diagnostic criteria for diabetes itself (HbA1c $\geq$ 6.5%, fasting glucose $\geq$ 126 mg/dL), not independent risk factors. Their inclusion means models trained on the `iammustafatz` dataset partly reproduce a known clinical threshold rather than discovering genuine lifestyle-based risk patterns - an important distinction from the `BRFSS` dataset, where no comparably direct diagnostic feature exists.

### Modeling Approach

**Logistic Regression** (interpretable baseline) and **Random Forest** (captures non-linear threshold effects) were trained on both datasets. For the larger `BRFSS` dataset, **Stacking** and **Voting** ensembles combining **Logistic Regression**, **Random Forest**, and **HistGradientBoosting** were additionally tested, given the larger sample size better supports the added model complexity. **SVM** was considered but rejected due to poor scalability at this sample size. All models used `class_weight="balanced"` to address class imbalance (`BRFSS`: ~14% positive; `iammustafatz`: ~8.8% positive), rather than oversampling/undersampling, keeping the training data distribution unmodified. Stratified 80/20 train/test splits and `StratifiedKFold` cross-validation were used throughout to ensure stable, representative evaluation given the class imbalance.

### Results Summary

| Dataset | Best Model | ROC-AUC | F1 | PR-AUC |
|---|---|---|---|---|
| `BRFSS` | Stacking Ensemble | 0.8252 | 0.4734 | 0.4565 |
| `iammustafatz` | Random Forest | 0.9746 | 0.6129 | 0.8757 |

The substantial performance gap between the two datasets is expected and directly connects to the data quality finding above: `HbA1c_level` and `blood_glucose_level` give the `iammustafatz` models a much stronger, more direct signal than the purely lifestyle/demographic `BRFSS` features. This gap should be read as a reflection of feature set composition, not as evidence that one dataset or model is inherently "better."

For `BRFSS`, all four models (**Logistic Regression**, **Random Forest**, **Stacking**, **Voting**) performed similarly (ROC-AUC 0.818–0.825), suggesting most of the available predictive signal is captured even by the simplest model. The **Stacking** ensemble's marginal edge over **Voting** (0.0009 ROC-AUC) suggests its meta-learner adds only limited value beyond a simple averaged combination. `general_health`, `high_blood_pressure`, `bmi`, `high_cholesterol`, and `age` consistently emerged as the strongest predictors across both **Logistic Regression** and **Random Forest**, reinforcing confidence that these reflect genuine clinical signal.

Overfitting checks (train vs. test ROC-AUC gap) were small across all models in both datasets (`BRFSS`: 0.010–0.016; `iammustafatz`: comparably small), indicating that hyperparameter tuning successfully controlled model complexity without memorizing the training data.

### Limitations

- **Correlational, not causal**: all reported relationships (e.g. `chol_checked_recently`, income, education) describe statistical association within this data, not causal mechanisms.
- **Selection bias in `BRFSS`**: the original data preparation removes respondents who declined to answer any of ~20 survey questions, systematically excluding people less willing to disclose sensitive information (income, weight, health status).
- **Missing established risk factors**: `_RACE`, an established diabetes risk factor, is absent from the original `BRFSS` feature selection and was not added in this analysis; kidney disease and depression history were identified as other plausible omissions.
- **Production generalization**: neither **RobustScaler** nor **StandardScaler** can guard against future input values outside the observed training range; this would require explicit input validation in a deployed setting, out of scope here.
- **Heterogeneous merged target class**: the merged "at-risk" class in `BRFSS` (prediabetes + diabetes) is not a clinically uniform group, though the evidence supports this grouping over the alternative.

### Future Work

Possible extensions include: adding `_RACE` and other omitted risk factors to the `BRFSS` feature set; training an `iammustafatz` model restricted to lifestyle/demographic features only (excluding HbA1c/glucose) to obtain a more realistic early-screening estimate, isolated from the diagnostic circularity discussed above;

## References

### Datasets

[Diabetes Health Indicators Dataset](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)

[Behavioral Risk Factor Surveillance System](https://www.kaggle.com/datasets/cdc/behavioral-risk-factor-surveillance-system)

[Diabetes prediction dataset](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset)

### Academic References

[Diabetes](https://www.mayoclinic.org/diseases-conditions/diabetes/symptoms-causes/syc-20371444)

[Prediabetes](https://www.mayoclinic.org/diseases-conditions/prediabetes/symptoms-causes/syc-20355278)